## Loading and discovering dataset

In [2]:
import pandas as pd

df = pd.read_csv('twitter_emotion.csv')

df = df.drop(columns=["Unnamed: 0"])

label_map = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

df["label_name"] = df["label"].map(label_map)

In [3]:
df['text'].sample(10)

331318    im wracked with feeling foolish and like a tot...
402571    im feeling nervous and terrified but also a bi...
166395                  im sorry if someone feels disturbed
61501     im left feeling sorry for my children that the...
155487    i feel like this has been a really lame update...
63870                          i woke up feeling very funny
13230                    i was afraid of feeling vulnerable
26768     im tired of feeling like crap and i dont think...
166664    i need to know if you are feeling shitty so th...
293767    i feel defective unable to control my own body...
Name: text, dtype: object

In [4]:
df['label'].value_counts()

label
1    141067
0    121187
3     57317
4     47712
2     34554
5     14972
Name: count, dtype: int64

In [5]:
df["text_length"] = df["text"].apply(lambda x: len(x.split()))

df["text_length"].describe()

count    416809.000000
mean         19.211015
std          11.051049
min           1.000000
25%          11.000000
50%          17.000000
75%          25.000000
max         178.000000
Name: text_length, dtype: float64

In [6]:
df.sort_values("text_length", ascending=False).head(5)

,text,label,label_name,text_length
348392,a few days back i was waiting for the bus at t...,3,anger,178
291479,two years back someone invited me to be the tu...,3,anger,110
97990,i have been thinking of changing my major for ...,0,sadness,101
22827,my living and working conditions at home were ...,0,sadness,100
38708,when i got into a bus i found that my wallet h...,3,anger,100


In [7]:
df.sort_values("text_length").head(5)

,text,label,label_name,text_length
28,when,3,anger,1
151491,never,4,fear,1
149927,spoilt,3,anger,1
303887,sadness,0,sadness,1
204577,when,4,fear,1


In [8]:
def summarize_text_cleanliness(text):
  summary = ""
  urls_count = df[text].str.contains(r"http\S+", regex=True).sum()
  summary += f"URLs Count: {urls_count}\n"

  hashtags_count = df[text].str.contains(r"#\w+", regex=True).sum()
  summary += f"Hashtags Count: {hashtags_count}\n"

  mentions_count = df[text].str.contains(r"@\w+", regex=True).sum()
  summary += f"Mentions Count: {mentions_count}\n"

  punctuation_count = df[text].str.contains(r"[^\w\s]", regex=True).sum()
  summary += f"Punctuation Count: {punctuation_count}\n"

  multiple_spaces_count = df[text].str.contains(r"\s{2,}", regex=True).sum()
  summary += f"Multiple Spaces Count: {multiple_spaces_count}\n"

  leading_trailing_spaces_count = df[text].str.startswith(" ").sum() + df[text].str.endswith(" ").sum()
  summary += f"Leading/Trailing Spaces Count: {leading_trailing_spaces_count}\n"

  uppercase_count = df[text].str.contains(r"[A-Z]", regex=True).sum()
  summary += f"Uppercase Letters Count: {uppercase_count}\n"

  return summary

cleanliness_summary = summarize_text_cleanliness('text')
print(cleanliness_summary)

URLs Count: 150
Hashtags Count: 0
Mentions Count: 0
Punctuation Count: 0
Multiple Spaces Count: 0
Leading/Trailing Spaces Count: 0
Uppercase Letters Count: 0



- As we saw that the dataset contained no punctuation, mentions, or hashtags, and only a negligible number of URLs (~150). Therefore, we will apply simple cleaning

In [9]:
import re

def remove_urls(tweet):
  tweet = re.sub(r'http\S+', '', tweet)
  tweet = re.sub(r'\s+', ' ', tweet)
  return tweet.strip()

df['cleaned_text'] = df['text'].apply(remove_urls)

In [10]:
print(summarize_text_cleanliness('cleaned_text'))

URLs Count: 0
Hashtags Count: 0
Mentions Count: 0
Punctuation Count: 0
Multiple Spaces Count: 0
Leading/Trailing Spaces Count: 0
Uppercase Letters Count: 0



## Tokenization

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=100000, oov_token="<OOV>")

tokenizer.fit_on_texts(df['cleaned_text'])

sequences = tokenizer.texts_to_sequences(df['cleaned_text'])

word_index = tokenizer.word_index

print(len(word_index))
print(list(word_index.items())[:10])
print(sequences[0])

75300
[('<OOV>', 1), ('i', 2), ('feel', 3), ('and', 4), ('to', 5), ('the', 6), ('a', 7), ('feeling', 8), ('that', 9), ('of', 10)]
[2, 32, 3, 41, 286, 4, 1501, 3119]


## Padding

In [12]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def pad_text_sequences(sequences, maxlen):
  padded = pad_sequences(sequences, maxlen=maxlen, padding='post', truncating='post')

  return padded
    
padded_sequences = pad_text_sequences(sequences, maxlen=50)

print(padded_sequences.shape)
print(padded_sequences[0])

(416809, 50)
[   2   32    3   41  286    4 1501 3119    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]


## Data Preparation

In [13]:
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

def prepare_data_for_training(padded_sequences, labels, test_size):
  num_classes = len(set(labels))

  labels = labels.values

  y = to_categorical(labels, num_classes=num_classes)

  X_train, X_test, y_train, y_test = train_test_split(padded_sequences, y, test_size=test_size, random_state=42, stratify=labels)

  return X_train, X_test, y_train, y_test

## GloVe Model

In [14]:
import gensim.downloader as api

glove_model = api.load("glove-twitter-50")

print(glove_model['happy'])
print(glove_model['happy'].shape)
print(glove_model.most_similar("happy", topn=5))

[-0.1614     0.2266     0.039414   0.33642   -0.62881   -0.1035
  0.89679    0.72547   -0.91096    0.066873  -0.88329   -0.10026
 -3.9851    -0.17096   -0.77985    0.74012   -0.57288   -0.23543
 -1.223      0.83262    0.15536    0.75707   -1.0698     0.75106
  0.24905    0.73248   -0.13189   -0.74685    1.558     -0.35085
 -0.41599   -0.22845    0.0049928  0.57969    0.15524    0.41605
  0.14427    0.23541    0.069321   0.86313   -2.2083    -0.95704
  0.57644    0.55662    0.077452   0.10875   -0.39615   -0.81149
 -0.43868    0.063696 ]
(50,)
[('birthday', 0.9478021860122681), ('day', 0.8840912580490112), ('love', 0.8704721331596375), ('thank', 0.8623552918434143), ('wish', 0.8603641390800476)]


## Embeddings

In [16]:
import numpy as np

def build_embedding_matrix(word_index, embedding_model, vector_size, num_words):
    vocab_size = len(word_index) + 1
    embedding_matrix = np.zeros((vocab_size, vector_size))
    
    for word, i in word_index.items():
        if i < num_words:
            if word in embedding_model:
                embedding_matrix[i] = embedding_model[word]
    
    return embedding_matrix

vocab_size = len(word_index) + 1
embedding_dim = 50
embedding_matrix = build_embedding_matrix(word_index, glove_model, embedding_dim, num_words=100000)
print(embedding_matrix.shape)
print(embedding_matrix[word_index['happy']])

(75301, 50)
[-0.16140001  0.22660001  0.039414    0.33642    -0.62880999 -0.1035
  0.89679003  0.72547001 -0.91096002  0.066873   -0.88328999 -0.10026
 -3.98510003 -0.17095999 -0.77985001  0.74011999 -0.57288003 -0.23543
 -1.22300005  0.83262002  0.15536     0.75707    -1.06980002  0.75106001
  0.24905001  0.73247999 -0.13189    -0.74685001  1.55799997 -0.35084999
 -0.41599    -0.22845     0.0049928   0.57968998  0.15524     0.41604999
  0.14427     0.23541     0.069321    0.86312997 -2.20830011 -0.95704001
  0.57643998  0.55662     0.077452    0.10875    -0.39614999 -0.81149
 -0.43867999  0.063696  ]


## Building & Training The Model

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, Flatten, Dense, Dropout

def build_cnn_model(vocab_size, embedding_matrix, input_length, trainable, num_classes):
  model = Sequential()

  model.add(Embedding(input_dim=vocab_size, output_dim=embedding_matrix.shape[1], weights=[embedding_matrix], input_length=input_length, trainable=trainable))

  model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))

  model.add(MaxPooling1D(pool_size=2))

  model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))

  model.add(Flatten())

  model.add(Dense(64, activation='relu'))

  model.add(Dropout(0.5))

  model.add(Dense(num_classes, activation='softmax'))

  model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

  return model

In [18]:
def train_model(model, X_train, y_train, epochs, batch_size):
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.1)
    return history

## CNN with GloVe Embeddings Using Test Size 0.2

In [42]:
embedding_matrix_glove = build_embedding_matrix(word_index, glove_model, vector_size=50 ,num_words=100000)

X_train, X_test, y_train, y_test = prepare_data_for_training(padded_sequences, df['label'], test_size=0.2)

model_glove = build_cnn_model(vocab_size=len(word_index) + 1, embedding_matrix=embedding_matrix_glove, input_length=50, trainable=False, num_classes=6)

print(model_glove.summary())

history_glove = train_model(model_glove, X_train, y_train, epochs=5, batch_size=128)

loss_glove, accuracy_glove = model_glove.evaluate(X_test, y_test)
print(f"GloVe Model - Loss: {loss_glove:.4f}, Accuracy: {accuracy_glove:.4f}")

g:\FCAI\NLP\Assignments\NLP_Project\.venv310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │     3,765,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,765,050 (14.36 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 3,765,050 (14.36 MB)

None
Epoch 1/5
2345/2345 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.7431 - loss: 0.7066 - val_accuracy: 0.8833 - val_loss: 0.2975
Epoch 2/5
2345/2345 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.8950 - loss: 0.2651 - val_accuracy: 0.9023 - val_loss: 0.2192
Epoch 3/5
2345/2345 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.9115 - loss: 0.1996 - val_accuracy: 0.9109 - val_loss: 0.1870
Epoch 4/5
2345/2345 ━━━━━━━━━━━━━━━━━━━━ 27s 11ms/step - accuracy: 0.9177 - loss: 0.1730 - val_accuracy: 0.9128 - val_loss: 0.1875
Epoch 5/5
2345/2345 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.9219 - loss: 0.1576 - val_accuracy: 0.9176 - val_loss: 0.1801
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9138 - loss: 0.1880
GloVe Model - Loss: 0.1880, Accuracy: 0.9138


## CNN with GloVe Embeddings Using Test Size 0.3

In [19]:
embedding_matrix_glove = build_embedding_matrix(word_index, glove_model, vector_size=50 ,num_words=100000)

X_train, X_test, y_train, y_test = prepare_data_for_training(padded_sequences, df['label'], test_size=0.3)

model_glove = build_cnn_model(vocab_size=len(word_index) + 1, embedding_matrix=embedding_matrix_glove, input_length=50, trainable=False, num_classes=6)

print(model_glove.summary())

history_glove = train_model(model_glove, X_train, y_train, epochs=5, batch_size=128)

loss_glove, accuracy_glove = model_glove.evaluate(X_test, y_test)
print(f"GloVe Model - Loss: {loss_glove:.4f}, Accuracy: {accuracy_glove:.4f}")

g:\FCAI\NLP\Assignments\NLP_Project\.venv310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     3,765,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,765,050 (14.36 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 3,765,050 (14.36 MB)

None
Epoch 1/5
2052/2052 ━━━━━━━━━━━━━━━━━━━━ 30s 14ms/step - accuracy: 0.7302 - loss: 0.7388 - val_accuracy: 0.8744 - val_loss: 0.3266
Epoch 2/5
2052/2052 ━━━━━━━━━━━━━━━━━━━━ 27s 13ms/step - accuracy: 0.8932 - loss: 0.2758 - val_accuracy: 0.9014 - val_loss: 0.2220
Epoch 3/5
2052/2052 ━━━━━━━━━━━━━━━━━━━━ 30s 15ms/step - accuracy: 0.9111 - loss: 0.2055 - val_accuracy: 0.9079 - val_loss: 0.2031
Epoch 4/5
2052/2052 ━━━━━━━━━━━━━━━━━━━━ 28s 14ms/step - accuracy: 0.9185 - loss: 0.1765 - val_accuracy: 0.9063 - val_loss: 0.2214
Epoch 5/5
2052/2052 ━━━━━━━━━━━━━━━━━━━━ 31s 15ms/step - accuracy: 0.9224 - loss: 0.1593 - val_accuracy: 0.9118 - val_loss: 0.1916
3908/3908 ━━━━━━━━━━━━━━━━━━━━ 26s 7ms/step - accuracy: 0.9096 - loss: 0.1946
GloVe Model - Loss: 0.1946, Accuracy: 0.9096


## CNN with Word2Vec Embeddings Using Test Size 0.2

In [43]:
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec


tokenized_texts = [simple_preprocess(text) for text in df['cleaned_text']]

w2v_model = Word2Vec(sentences=tokenized_texts, vector_size=50, window=5, min_count=2, workers=4)

embedding_matrix_w2v = build_embedding_matrix(word_index, w2v_model.wv, vector_size=50, num_words=100000)

X_train, X_test, y_train, y_test = prepare_data_for_training(padded_sequences, df['label'], test_size=0.2)

model_w2v = build_cnn_model(vocab_size=len(word_index) + 1, embedding_matrix=embedding_matrix_w2v, input_length=50, trainable=True, num_classes=6)

print(model_w2v.summary())

history_w2v = train_model(model_w2v, X_train, y_train, epochs=5, batch_size=128)

loss_w2v, accuracy_w2v = model_w2v.evaluate(X_test, y_test)
print(f"Word2Vec Model - Loss: {loss_w2v:.4f}, Accuracy: {accuracy_w2v:.4f}")

g:\FCAI\NLP\Assignments\NLP_Project\.venv310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │     3,765,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,765,050 (14.36 MB)

 Trainable params: 3,765,050 (14.36 MB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/5
2345/2345 ━━━━━━━━━━━━━━━━━━━━ 55s 23ms/step - accuracy: 0.8505 - loss: 0.3959 - val_accuracy: 0.9293 - val_loss: 0.1425
Epoch 2/5
2345/2345 ━━━━━━━━━━━━━━━━━━━━ 78s 21ms/step - accuracy: 0.9295 - loss: 0.1414 - val_accuracy: 0.9310 - val_loss: 0.1192
Epoch 3/5
2345/2345 ━━━━━━━━━━━━━━━━━━━━ 54s 23ms/step - accuracy: 0.9335 - loss: 0.1219 - val_accuracy: 0.9335 - val_loss: 0.1166
Epoch 4/5
2345/2345 ━━━━━━━━━━━━━━━━━━━━ 51s 22ms/step - accuracy: 0.9357 - loss: 0.1128 - val_accuracy: 0.9329 - val_loss: 0.1140
Epoch 5/5
2345/2345 ━━━━━━━━━━━━━━━━━━━━ 52s 22ms/step - accuracy: 0.9369 - loss: 0.1080 - val_accuracy: 0.9332 - val_loss: 0.1142
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9323 - loss: 0.1213
Word2Vec Model - Loss: 0.1213, Accuracy: 0.9323


## CNN with Word2Vec Embeddings Using Test Size 0.3

In [20]:
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec


tokenized_texts = [simple_preprocess(text) for text in df['cleaned_text']]

w2v_model = Word2Vec(sentences=tokenized_texts, vector_size=50, window=5, min_count=2, workers=4)

embedding_matrix_w2v = build_embedding_matrix(word_index, w2v_model.wv, vector_size=50, num_words=100000)

X_train, X_test, y_train, y_test = prepare_data_for_training(padded_sequences, df['label'], test_size=0.3)

model_w2v = build_cnn_model(vocab_size=len(word_index) + 1, embedding_matrix=embedding_matrix_w2v, input_length=50, trainable=True, num_classes=6)

print(model_w2v.summary())

history_w2v = train_model(model_w2v, X_train, y_train, epochs=5, batch_size=128)

loss_w2v, accuracy_w2v = model_w2v.evaluate(X_test, y_test)
print(f"Word2Vec Model - Loss: {loss_w2v:.4f}, Accuracy: {accuracy_w2v:.4f}")

g:\FCAI\NLP\Assignments\NLP_Project\.venv310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │     3,765,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,765,050 (14.36 MB)

 Trainable params: 3,765,050 (14.36 MB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/5
2052/2052 ━━━━━━━━━━━━━━━━━━━━ 54s 25ms/step - accuracy: 0.8456 - loss: 0.4136 - val_accuracy: 0.9282 - val_loss: 0.1475
Epoch 2/5
2052/2052 ━━━━━━━━━━━━━━━━━━━━ 53s 26ms/step - accuracy: 0.9301 - loss: 0.1451 - val_accuracy: 0.9308 - val_loss: 0.1302
Epoch 3/5
2052/2052 ━━━━━━━━━━━━━━━━━━━━ 52s 25ms/step - accuracy: 0.9341 - loss: 0.1238 - val_accuracy: 0.9327 - val_loss: 0.1248
Epoch 4/5
2052/2052 ━━━━━━━━━━━━━━━━━━━━ 61s 30ms/step - accuracy: 0.9357 - loss: 0.1142 - val_accuracy: 0.9322 - val_loss: 0.1323
Epoch 5/5
2052/2052 ━━━━━━━━━━━━━━━━━━━━ 50s 24ms/step - accuracy: 0.9370 - loss: 0.1082 - val_accuracy: 0.9327 - val_loss: 0.1283
3908/3908 ━━━━━━━━━━━━━━━━━━━━ 27s 7ms/step - accuracy: 0.9308 - loss: 0.1313
Word2Vec Model - Loss: 0.1313, Accuracy: 0.9308


## Performance Comparison of Models

In [21]:
print("GloVe Accuracy Using Test Size 0.2: 0.9138" )
print("GloVe Accuracy Using Test Size 0.3: 0.9096" )
print("Word2Vec Accuracy Using Test Size 0.2: 0.9323" )
print("Word2Vec Accuracy Using Test Size 0.3: 0.9308" )

GloVe Accuracy Using Test Size 0.2: 0.9138
GloVe Accuracy Using Test Size 0.3: 0.9096
Word2Vec Accuracy Using Test Size 0.2: 0.9323
Word2Vec Accuracy Using Test Size 0.3: 0.9308


## Impact of Padding Length on Model Performance

In [ ]:
lengths = [50, 100, 500, 700, 1000]
results = []

for length in lengths:
    padded = pad_text_sequences(sequences, maxlen=length)

    X_train, X_test, y_train, y_test = prepare_data_for_training(padded, df['label'], test_size=0.2)

    X_train_small = X_train[:100000]
    y_train_small = y_train[:100000]

    model = build_cnn_model(vocab_size=vocab_size, embedding_matrix=embedding_matrix_w2v, input_length=length, trainable=True, num_classes=6 )

    history = train_model(model, X_train_small, y_train_small, epochs=2, batch_size=128 )

    val_acc = history.history['val_accuracy'][-1]
    results.append((length, val_acc))


print("Results:")
for length, acc in results:
    print(f"Max Length: {length}, Validation Accuracy: {acc:.4f}")

Epoch 1/2


g:\FCAI\NLP\Assignments\NLP_Project\.venv310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


704/704 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - accuracy: 0.7090 - loss: 0.7946 - val_accuracy: 0.9066 - val_loss: 0.2735
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 16s 22ms/step - accuracy: 0.9140 - loss: 0.2382 - val_accuracy: 0.9196 - val_loss: 0.1888
Epoch 1/2


g:\FCAI\NLP\Assignments\NLP_Project\.venv310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


704/704 ━━━━━━━━━━━━━━━━━━━━ 21s 28ms/step - accuracy: 0.7273 - loss: 0.7497 - val_accuracy: 0.8964 - val_loss: 0.2812
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 20s 29ms/step - accuracy: 0.9154 - loss: 0.2229 - val_accuracy: 0.9200 - val_loss: 0.1860
Epoch 1/2


...

704/704 ━━━━━━━━━━━━━━━━━━━━ 65s 90ms/step - accuracy: 0.7185 - loss: 0.7672 - val_accuracy: 0.8947 - val_loss: 0.3081
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 59s 84ms/step - accuracy: 0.9134 - loss: 0.2429 - val_accuracy: 0.9219 - val_loss: 0.1797


g:\FCAI\NLP\Assignments\NLP_Project\.venv310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 92s 129ms/step - accuracy: 0.7239 - loss: 0.7469 - val_accuracy: 0.9015 - val_loss: 0.2802
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 88s 125ms/step - accuracy: 0.9128 - loss: 0.2481 - val_accuracy: 0.9205 - val_loss: 0.1973
Epoch 1/2


g:\FCAI\NLP\Assignments\NLP_Project\.venv310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


704/704 ━━━━━━━━━━━━━━━━━━━━ 128s 180ms/step - accuracy: 0.7116 - loss: 0.7853 - val_accuracy: 0.8945 - val_loss: 0.3045
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 122s 173ms/step - accuracy: 0.9035 - loss: 0.2691 - val_accuracy: 0.9215 - val_loss: 0.1982
Results:
Max Length: 50, Validation Accuracy: 0.9196
Max Length: 100, Validation Accuracy: 0.9200
Max Length: 200, Validation Accuracy: 0.9214
Max Length: 500, Validation Accuracy: 0.9219
Max Length: 700, Validation Accuracy: 0.9205
Max Length: 1000, Validation Accuracy: 0.9215


## User Input Emotion Prediction

In [48]:
import numpy as np

user_input = input("Enter a tweet: ")

cleaned_input = remove_urls(user_input)

user_sequence = tokenizer.texts_to_sequences([cleaned_input])

padded = pad_text_sequences(user_sequence, maxlen=50)

prediction = model_w2v.predict(padded)

predicted_label = np.argmax(prediction)

print(f"Predicted Emotion: {label_map[predicted_label]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Predicted Emotion: joy
